# Run your experimental adapter on free Colab
Select **Runtime → Change runtime type → T4 GPU**, then **Run all**.
Upload `adapter_config.json` and `adapter_model.safetensors` when asked. Enter your Render API key privately when prompted. No Hugging Face PRO or token is needed.

This temporarily hosts your existing adapter; it does not train or validate it. Its earlier grading results were poor. Keep Claude available and treat this model as experimental. Colab may stop the session at any time; rerun this notebook to reconnect using a new temporary URL. Do not share a notebook with saved outputs. This notebook neither buys compute nor keeps a session alive artificially.


In [ ]:
import subprocess, sys
# Stop earlier notebook processes before starting a new hosting session.
for name in ("tunnel", "server"):
    process = globals().get(name)
    if process is not None and process.poll() is None:
        process.terminate()
        process.wait(timeout=30)
# This LoRA server uses ordinary PyTorch layers, not torchao quantization.
# Colab may bundle an old torchao that causes PEFT adapter loading to fail.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.46,<5', 'peft>=0.14,<1', 'accelerate>=1,<2',
    'fastapi>=0.115,<1', 'uvicorn>=0.30,<1', 'httpx>=0.27,<1'], check=True)
import torch
assert torch.cuda.is_available(), 'Choose a T4 GPU runtime, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import json, os, secrets
from google.colab import files
work = Path('/content/essaygrader-host')
work.mkdir(exist_ok=True)
adapter = work / 'adapter'
adapter.mkdir(exist_ok=True)
print('Upload BOTH adapter_config.json and adapter_model.safetensors.')
uploaded = files.upload()
for name in ['adapter_config.json', 'adapter_model.safetensors']:
    assert name in uploaded, 'Missing ' + name
    (adapter / name).write_bytes(uploaded[name])
del uploaded
config = json.loads((adapter / 'adapter_config.json').read_text())
assert config['base_model_name_or_path'] == 'Qwen/Qwen2.5-1.5B-Instruct'
server_key = secrets.token_urlsafe(48)
print('Adapter files ready. Server key generated privately.')


In [ ]:
import httpx, time
# Fetch the reviewed hosting server from a fixed repository commit.
revision = '41bd6128eb16bda81daa3f6fee332a64c992966b'
response = httpx.get(f'https://raw.githubusercontent.com/sfstast/essaygrader/{revision}/deploy/open-model/server.py', timeout=60)
response.raise_for_status()
(work / 'server.py').write_text(response.text)
env = dict(os.environ, MODEL_SERVER_KEY=server_key,
           ADAPTER_PATH=str(adapter), SERVED_MODEL_NAME='essaygrader-experimental')
log = open(work / 'server.log', 'w')
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'server:app',
    '--host', '0.0.0.0', '--port', '7860', '--no-access-log'],
    cwd=work, env=env, stdout=log, stderr=log)
for _ in range(900):
    if server.poll() is not None:
        raise RuntimeError('Model server stopped. Inspect /content/essaygrader-host/server.log locally.')
    try:
        health = httpx.get('http://127.0.0.1:7860/health', timeout=3)
        if health.is_success and health.json().get('adapter_loaded'):
            break
    except httpx.HTTPError:
        pass
    time.sleep(2)
else:
    raise TimeoutError('Model startup exceeded 30 minutes.')
# Verify actual generation before connecting the website.
response = httpx.post('http://127.0.0.1:7860/v1/chat/completions',
    headers={'Authorization': 'Bearer ' + server_key},
    json={'model': 'essaygrader-experimental', 'max_tokens': 32,
          'messages': [{'role': 'user', 'content': 'Give one short tip for supporting an essay claim with evidence.'}]}, timeout=120)
response.raise_for_status()
assert response.json()['choices'][0]['message']['content'].strip()
print('Adapter loaded and real inference passed.')


In [ ]:
import hashlib, re
release = httpx.get('https://api.github.com/repos/cloudflare/cloudflared/releases/latest', timeout=60)
release.raise_for_status()
asset = next(a for a in release.json()['assets'] if a['name'] == 'cloudflared-linux-amd64')
expected = asset.get('digest', '')
assert expected.startswith('sha256:'), 'Release has no checksum; stop rather than run an unverified binary.'
binary = httpx.get(asset['browser_download_url'], follow_redirects=True, timeout=120)
binary.raise_for_status()
assert hashlib.sha256(binary.content).hexdigest() == expected.split(':', 1)[1]
cloudflared = work / 'cloudflared'
cloudflared.write_bytes(binary.content)
cloudflared.chmod(0o700)
tunnel_log = work / 'tunnel.log'
tunnel = subprocess.Popen([str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:7860',
    '--no-autoupdate'], stdout=open(tunnel_log, 'w'), stderr=subprocess.STDOUT)
for _ in range(90):
    if tunnel.poll() is not None:
        raise RuntimeError('Tunnel stopped. Inspect tunnel.log locally.')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', tunnel_log.read_text())
    if match:
        model_url = match.group(0)
        break
    time.sleep(2)
else:
    raise TimeoutError('Tunnel did not start.')
for _ in range(30):
    try:
        response = httpx.get(model_url + '/health', timeout=10)
        if response.is_success:
            break
    except httpx.HTTPError:
        pass
    time.sleep(2)
else:
    raise RuntimeError('Temporary model URL is not reachable.')
print('Temporary endpoint:', model_url)


In [ ]:
import getpass
render_key = getpass.getpass('Render API key (hidden; never saved): ')
headers = {'Authorization': 'Bearer ' + render_key}
render_api = 'https://api.render.com/v1'
# This is the existing free website, not a new paid service.
service_id = 'srv-db8nshcs728c73adpqog'
response = httpx.get(f'{render_api}/services/{service_id}', headers=headers, timeout=30)
response.raise_for_status()
assert response.json()['serviceDetails']['url'] == 'https://reasoning-studio.onrender.com'
settings = {'OPEN_MODEL_BASE_URL': model_url + '/v1',
            'OPEN_MODEL_NAME': 'essaygrader-experimental', 'OPEN_MODEL_API_KEY': server_key}
# Set only these individual variables; preserve all other Render settings.
for key, value in settings.items():
    response = httpx.put(f'{render_api}/services/{service_id}/env-vars/{key}',
        headers=headers, json={'value': value}, timeout=30)
    response.raise_for_status()
response = httpx.post(f'{render_api}/services/{service_id}/deploys', headers=headers,
    json={'clearCache': 'do_not_clear'}, timeout=30)
response.raise_for_status()
deploy_id = response.json()['id']
for _ in range(120):
    response = httpx.get(f'{render_api}/services/{service_id}/deploys/{deploy_id}', headers=headers, timeout=30)
    response.raise_for_status()
    status = response.json()['status']
    if status == 'live':
        break
    if status in ['build_failed', 'update_failed', 'canceled', 'deactivated']:
        raise RuntimeError('Render deployment status: ' + status)
    time.sleep(10)
else:
    raise TimeoutError('Check deployment progress in the Render dashboard.')
headers.clear()
del render_key, settings
response = httpx.get('https://reasoning-studio.onrender.com/api/config', timeout=60)
response.raise_for_status()
provider = next(p for p in response.json()['providers'] if p['id'] == 'open_model')
assert provider['configured'] and provider.get('server_key_configured')
print('Connected! Open https://reasoning-studio.onrender.com/#workspace')
print('Select Open model / fine-tuned adapter. Leave the model server key blank.')
print('Keep this Colab runtime connected. Claude remains available if the runtime stops.')


## Stop hosting
Run the cell below only when you want to stop. Colab also stops hosting when its runtime disconnects. Render retains the old endpoint until the next notebook run updates it; Claude remains available.


In [ ]:
# Stop hosting when finished:
# tunnel.terminate()
# server.terminate()
